# Training dataset pipeline

Cleaned replacement for `1_training_dataset_csv.ipynb`. **Run all cells top-to-bottom.**

Builds the 695-image YOLO training inventory, GSVI↔SC sensitivity swap pools, label manifests, scenario folders, and held-out test sets.

**Export schema (slim):** `img_name`, `year`, `month`, `lat`, `lon`, `panoid`, `img_dir`, `exist`, `collection`, `class`, `metadata_status`

- `collection`: **GSVI** if `img_name` ends with `_0/_90/_180/_270`; otherwise **SC**
- `class`: **waste** if the YOLO label file is non-empty; otherwise **background**
- Split file has 696 lines; `.DS_Store` is filtered → 695 images
- 8 SC images have no row in `*_basic.csv` (metadata left blank, `metadata_status=missing`)


In [11]:
import math
import os
import re
import shutil
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display


In [12]:
DATA_ROOT = Path(os.environ.get("PHD_DATA_ROOT", "/Users/wenlanzhang/Downloads/PhD_UCL/Data"))

IMG_ROOT = DATA_ROOT / "Waste/img"
TRAIN_ROOT = IMG_ROOT / "Train0315_695"
OUTPUT_DIR = DATA_ROOT / "Chapter_waste" / "6_sensitivity"
SENSITIVITY_IMG_ROOT = IMG_ROOT / "sensitivity"

SPLIT_FILE = TRAIN_ROOT / "dataSet/trainval.txt"
IMAGES_DIR = TRAIN_ROOT / "images"
LABELS_DIR = TRAIN_ROOT / "labels"
BASIC_CSVS = [IMG_ROOT / f"{s}_basic.csv" for s in ("Google", "Faith", "ZWL")]
CORRECT_SVI_CSV = IMG_ROOT / "Correct_SVI.csv"
COMBINED_SVI_CSV = IMG_ROOT / "Combined_SVI.csv"

LABELED_SENSITIVITY_DIR = SENSITIVITY_IMG_ROOT / "20260708_sensitivity"
LABELED_WASTE_EXPORT_DIR = SENSITIVITY_IMG_ROOT / "20260708_label_waste"
LABELED_WASTE_DIR = LABELED_WASTE_EXPORT_DIR / "images"      # Label Studio export (waste batch)
LABELED_WASTE_LABEL_DIR = LABELED_WASTE_EXPORT_DIR / "labels"
LABELED_BG_DIR = LABELED_SENSITIVITY_DIR / "background"      # no labels — empty .txt written at build time

SENSITIVITY_WASTE_DIR = SENSITIVITY_IMG_ROOT / "waste"
SENSITIVITY_BG_DIR = SENSITIVITY_IMG_ROOT / "background"

OUTPUT_CSV = OUTPUT_DIR / "Train0315_695_training.csv"
SENSITIVITY_CSV = OUTPUT_DIR / "Train0315_695_sensitivity_inventory.csv"
SCENARIO_DIR = OUTPUT_DIR / "scenarios"
POOLS_SUMMARY_CSV = OUTPUT_DIR / "Train0315_695_pools_summary.csv"
LABEL_MANIFEST_CSV = OUTPUT_DIR / "Train0315_695_gsvi_label_manifest.csv"
SWAP_NEEDS_CSV = OUTPUT_DIR / "Train0315_695_gsvi_swap_needs.csv"

HELD_OUT_TAG = "20260714_heldout"
HELD_OUT_DIR = SENSITIVITY_IMG_ROOT / HELD_OUT_TAG
HELD_OUT_MANIFEST_CSV = HELD_OUT_DIR / "held_out_manifest.csv"
HELD_OUT_CSV = OUTPUT_DIR / "Train0315_695_held_out_test.csv"
HELD_OUT_UNITS_CSV = OUTPUT_DIR / "held_out_test_units.csv"

HELD_OUT_GSVI_TAG = "20260714_heldout_GSVI"
HELD_OUT_GSVI_DIR = SENSITIVITY_IMG_ROOT / HELD_OUT_GSVI_TAG
HELD_OUT_GSVI_MANIFEST_CSV = HELD_OUT_GSVI_DIR / "held_out_manifest.csv"
HELD_OUT_GSVI_CSV = OUTPUT_DIR / "Train0315_695_held_out_gsvi_test.csv"
HELD_OUT_GSVI_UNITS_CSV = OUTPUT_DIR / "held_out_gsvi_test_units.csv"
PRIOR_HELD_OUT_CSV = HELD_OUT_CSV

SC_SHARE_PCTS = [0, 25, 50, 75, 100]
RANDOM_SEED = 42
LABEL_BUFFER_PCT = 0.40
WASTE_SHARE = 0.5
HELD_OUT_SEED = 42
GSVI_WASTE_SHARE = 0.5
HELD_OUT_GSVI_SEED = 142
EXCLUDE_PRIOR_HELD_OUT = False

COPY_IMAGES = False
BUILD_SCENARIOS = True
BUILD_HELD_OUT = True
BUILD_HELD_OUT_GSVI = True

SC_METADATA_OVERRIDES: dict[str, dict] = {}


## 1. Load the training image list


In [13]:
with open(SPLIT_FILE) as f:
    split_paths = [line.strip() for line in f if line.strip()]

raw_names = [Path(p).stem for p in split_paths]
junk = [n for n in raw_names if n == ".DS_Store"]
img_names = [n for n in raw_names if n != ".DS_Store"]

print(f"Split: {len(raw_names)} lines → {len(img_names)} images", end="")
if junk:
    print(f" (skipped junk: {junk})")
else:
    print()
print("Examples:", img_names[:5])


Split: 696 lines → 695 images (skipped junk: ['.DS_Store'])
Examples: ['LkN9dmH-f_QJFFxGOk7PEA_90', 'VSNF3NE5eykDgUUAqoayFg_270', '459eb9b0-UPoDfWE8S-Ya6uSXDf3w8Q_0', '701ab709-UrRoUBKATo0lZ4iSFwxUng_0', 'f10f8795-eIZ7kN3HHMhLCIFz3M5mNw_0']


## 2. Load metadata lookup tables


In [14]:
meta_parts = []
for path in BASIC_CSVS:
    part = pd.read_csv(path)
    part["meta_source"] = path.stem.replace("_basic", "")
    meta_parts.append(part)

meta = pd.concat(meta_parts, ignore_index=True)
meta_by_name = meta.set_index("img_name", drop=False)
meta_by_panoid = meta.dropna(subset=["panoid"]).drop_duplicates("panoid", keep="first").set_index("panoid", drop=False)

print(meta["meta_source"].value_counts())
meta.head(3)


meta_source
Google    453364
ZWL         2930
Faith        817
Name: count, dtype: int64


,panoid,lat,lon,year,month,img_name,img_dir,exist,day,hour,meta_source
0,khlXWXsuIX7rodKwi1fDAw,-1.347984,36.904885,2021,10,khlXWXsuIX7rodKwi1fDAw_0,Google/k/h/,True,NaN,NaN,Google
1,khlXWXsuIX7rodKwi1fDAw,-1.347984,36.904885,2021,10,khlXWXsuIX7rodKwi1fDAw_90,Google/k/h/,True,NaN,NaN,Google
2,khlXWXsuIX7rodKwi1fDAw,-1.347984,36.904885,2021,10,khlXWXsuIX7rodKwi1fDAw_180,Google/k/h/,True,NaN,NaN,Google


## 3. Helper functions


In [15]:
UUID_PREFIX = re.compile(r"^[0-9a-f]{8}-", re.I)
GSVI_SUFFIX = re.compile(r"_(0|90|180|270)$")


def collection_from_name(img_name: str) -> str:
    return "GSVI" if GSVI_SUFFIX.search(img_name) else "SC"


def collection_from_img_dir(img_dir) -> str | None:
    if pd.isna(img_dir):
        return None
    path = str(img_dir)
    if path.startswith("Google/"):
        return "GSVI"
    if path.startswith("Faith/") or path.startswith("ZWL/"):
        return "SC"
    return None


def candidate_names(img_name: str) -> list[str]:
    names = [img_name]
    if UUID_PREFIX.match(img_name):
        names.append(img_name.split("-", 1)[1])
    return list(dict.fromkeys(names))


def canonical_img_name(stem: str) -> str:
    return stem.split("-", 1)[1] if UUID_PREFIX.match(stem) else stem


def panoid_from_name(img_name: str) -> str:
    base = img_name.split("-", 1)[1] if UUID_PREFIX.match(img_name) else img_name
    if "_" in base:
        return base.rsplit("_", 1)[0]
    return base


def lookup_metadata(img_name: str) -> dict | None:
    for cand in candidate_names(img_name):
        if cand in meta_by_name.index:
            row = meta_by_name.loc[cand]
            return row.iloc[0].to_dict() if isinstance(row, pd.DataFrame) else row.to_dict()
    panoid = panoid_from_name(img_name)
    if panoid in meta_by_panoid.index:
        return meta_by_panoid.loc[panoid].to_dict()
    return None


def has_waste_label(img_name: str) -> bool:
    label_path = LABELS_DIR / f"{img_name}.txt"
    return label_path.exists() and label_path.stat().st_size > 0


def archive_image_path(img_name: str, img_dir: str) -> Path:
    return (IMG_ROOT / str(img_dir) / f"{img_name}.jpg").resolve()


def build_name_lookup(names: pd.Series) -> dict[str, str]:
    lookup: dict[str, str] = {}
    for canonical in names:
        for cand in candidate_names(canonical):
            lookup.setdefault(cand, canonical)
    return lookup


def match_lookup_name(img_name: str, lookup: dict[str, str]) -> str | None:
    for cand in candidate_names(img_name):
        if cand in lookup:
            return lookup[cand]
    return None


def has_local_files(img_name: str) -> bool:
    return (IMAGES_DIR / f"{img_name}.jpg").exists() and (LABELS_DIR / f"{img_name}.txt").exists()


def buffered_count(n: int, buffer_pct: float) -> int:
    return math.ceil(n * (1 + buffer_pct))


## 4. Build the training table


In [16]:
rows = []
missing_meta = []
missing_image = []

for img_name in img_names:
    image_path = IMAGES_DIR / f"{img_name}.jpg"
    if not image_path.exists():
        missing_image.append(img_name)

    meta_row = lookup_metadata(img_name)
    if meta_row is None and img_name in SC_METADATA_OVERRIDES:
        meta_row = SC_METADATA_OVERRIDES[img_name]

    metadata_status = "matched" if meta_row is not None else "missing"
    if meta_row is None:
        missing_meta.append(img_name)
        meta_row = {"year": pd.NA, "month": pd.NA, "lat": pd.NA, "lon": pd.NA, "panoid": pd.NA, "img_dir": pd.NA, "exist": False}

    rows.append({
        "img_name": img_name,
        "year": meta_row.get("year"),
        "month": meta_row.get("month"),
        "lat": meta_row.get("lat"),
        "lon": meta_row.get("lon"),
        "panoid": meta_row.get("panoid"),
        "img_dir": meta_row.get("img_dir"),
        "exist": image_path.exists() or bool(meta_row.get("exist", False)),
        "collection": collection_from_name(img_name),
        "class": "waste" if has_waste_label(img_name) else "background",
        "metadata_status": metadata_status,
    })

train_df = pd.DataFrame(rows)
TOTAL_IMAGES = len(train_df)
WASTE_TOTAL = int((train_df["class"] == "waste").sum())
BG_TOTAL = int((train_df["class"] == "background").sum())
TEST_TARGET_IMAGES = round(TOTAL_IMAGES * 0.15)
GSVI_TEST_TARGET_IMAGES = TEST_TARGET_IMAGES

print(f"Rows built: {TOTAL_IMAGES}")
print(train_df["collection"].value_counts().to_string())
print(train_df["class"].value_counts().to_string())
print(f"Missing metadata: {len(missing_meta)}")
print(f"Missing local image file: {len(missing_image)}")
if missing_meta:
    print("Unmatched:", missing_meta[:10])
train_df.head()


Rows built: 695
collection
GSVI    610
SC       85
class
waste         353
background    342
Missing metadata: 8
Missing local image file: 0
Unmatched: ['1f7e5b1e-3767618616712176', '12fe849c-WechatIMG1048', 'b177368d-WechatIMG1047', 'e46a1e51-WechatIMG1046', '44c2da02-IMG_6653', 'aa90b6e2-WechatIMG1049', 'dddd727b-IMG_6652', '37eb2376-310211603998748']


,img_name,year,month,lat,lon,panoid,img_dir,exist,collection,class,metadata_status
0,LkN9dmH-f_QJFFxGOk7PEA_90,2018,10,-1.374241,36.779327,LkN9dmH-f_QJFFxGOk7PEA,Google/L/k/,True,GSVI,background,matched
1,VSNF3NE5eykDgUUAqoayFg_270,2021,7,-1.238584,36.851001,VSNF3NE5eykDgUUAqoayFg,Google/V/S/,True,GSVI,background,matched
2,459eb9b0-UPoDfWE8S-Ya6uSXDf3w8Q_0,2018,4,-1.320441,36.892761,UPoDfWE8S-Ya6uSXDf3w8Q,Google/U/P/,True,GSVI,waste,matched
3,701ab709-UrRoUBKATo0lZ4iSFwxUng_0,2021,8,-1.312264,36.857056,UrRoUBKATo0lZ4iSFwxUng,Google/U/r/,True,GSVI,waste,matched
4,f10f8795-eIZ7kN3HHMhLCIFz3M5mNw_0,2021,9,-1.186151,36.905804,eIZ7kN3HHMhLCIFz3M5mNw,Google/e/I/,True,GSVI,waste,matched


## 5. Summary, QA & export


In [17]:
n = len(train_df)
counts = pd.crosstab(train_df["collection"], train_df["class"]).reindex(index=["GSVI", "SC"], columns=["background", "waste"])

train_df["collection_from_dir"] = train_df["img_dir"].map(collection_from_img_dir)
known = train_df["collection_from_dir"].notna()
mismatches = train_df.loc[known & (train_df["collection"] != train_df["collection_from_dir"])]
missing_dir = train_df[train_df["collection_from_dir"].isna()]

display(Markdown(f"### Training set — **{n:,}** images"))
display(counts)
print(f"Mismatches: {len(mismatches):,} | Missing img_dir: {len(missing_dir):,}")
if len(missing_dir):
    display(missing_dir[["img_name", "collection", "metadata_status"]].reset_index(drop=True))

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
export_df = train_df.drop(columns=["collection_from_dir"], errors="ignore")
export_df.to_csv(OUTPUT_CSV, index=False)
print(f"Saved {len(export_df):,} rows to {OUTPUT_CSV}")


### Training set — **695** images

class,background,waste
collection,,
GSVI,295,315
SC,47,38


Mismatches: 0 | Missing img_dir: 8


,img_name,collection,metadata_status
0,1f7e5b1e-3767618616712176,SC,missing
1,12fe849c-WechatIMG1048,SC,missing
2,b177368d-WechatIMG1047,SC,missing
3,e46a1e51-WechatIMG1046,SC,missing
4,44c2da02-IMG_6653,SC,missing
5,aa90b6e2-WechatIMG1049,SC,missing
6,dddd727b-IMG_6652,SC,missing
7,37eb2376-310211603998748,SC,missing


Saved 695 rows to /Users/wenlanzhang/Downloads/PhD_UCL/Data/Chapter_waste/6_sensitivity/Train0315_695_training.csv


## 6. GSVI ↔ SC swap pools


In [18]:
def sensitivity_role(in_training: bool, training_class: str, svi_waste: bool) -> str:
    if in_training and training_class == "waste" and svi_waste:
        return "training_waste_confirmed_svi"
    if in_training and training_class == "waste":
        return "training_waste_only"
    if in_training:
        return "training_background"
    return "svi_unused_waste"


def sc_slot_counts(sc_share_pct: int) -> dict[str, int]:
    sc_waste_keep = round(SC_WASTE_SLOTS * sc_share_pct / 100)
    sc_bg_keep = round(SC_BG_SLOTS * sc_share_pct / 100)
    return {
        "sc_waste_keep": sc_waste_keep,
        "sc_waste_swap": SC_WASTE_SLOTS - sc_waste_keep,
        "sc_bg_keep": sc_bg_keep,
        "sc_bg_swap": SC_BG_SLOTS - sc_bg_keep,
        "sc_total_keep": sc_waste_keep + sc_bg_keep,
        "sc_total_swap": (SC_WASTE_SLOTS - sc_waste_keep) + (SC_BG_SLOTS - sc_bg_keep),
    }


def pick_training_rows(df: pd.DataFrame, n: int, seed: int) -> pd.DataFrame:
    if n == 0:
        return df.iloc[0:0].copy()
    if n >= len(df):
        return df.copy()
    return df.sample(n=n, random_state=seed).copy()


def _pool_error_hint(pool_name: str) -> str:
    if "labeled_gsvi_waste" in pool_name or pool_name == "gsvi_waste_required":
        return (
            f"  Waste swaps need non-empty labels in: {LABELED_WASTE_LABEL_DIR}\n"
            f"  Images should be in: {LABELED_WASTE_DIR} (Label Studio export)"
        )
    if "labeled_gsvi_background" in pool_name or pool_name == "gsvi_background_required":
        return f"  Background swaps need images in: {LABELED_BG_DIR}"
    return ""


def pick_from_pool(pool: pd.DataFrame, n: int, seed: int, pool_name: str) -> pd.DataFrame:
    if n > len(pool):
        hint = _pool_error_hint(pool_name)
        raise ValueError(
            f"Need {n:,} images from '{pool_name}', but only {len(pool):,} available.\n{hint}"
        )
    local = pool[pool["has_local"]]
    non_local = pool[~pool["has_local"]]
    if n <= len(local):
        return local.sample(n=n, random_state=seed).copy()
    picked = [local]
    remaining = n - len(local)
    if remaining > len(non_local):
        hint = _pool_error_hint(pool_name)
        raise ValueError(
            f"Need {remaining:,} extra non-local images from '{pool_name}', "
            f"but only {len(non_local):,} available.\n{hint}"
        )
    picked.append(non_local.sample(n=remaining, random_state=seed))
    return pd.concat(picked, ignore_index=True)


def build_candidate_pool(collection: str, cls: str, training_df: pd.DataFrame, extras_df: pd.DataFrame) -> pd.DataFrame:
    train_part = training_df[(training_df["collection"] == collection) & (training_df["class"] == cls)].copy()
    train_part["candidate_source"] = "training"
    extra_part = extras_df[(extras_df["collection"] == collection) & (extras_df["class"] == cls)].copy()
    extra_part["candidate_source"] = "extra"
    pool = pd.concat([train_part, extra_part], ignore_index=True)
    pool = pool.drop_duplicates("img_name", keep="first")
    pool["has_local"] = pool["img_name"].map(has_local_files)
    return pool.sort_values(["has_local", "candidate_source"], ascending=[False, True])


svi_waste = pd.read_csv(CORRECT_SVI_CSV)
svi_waste["collection"] = svi_waste["img_name"].map(collection_from_name)
svi_waste.loc[svi_waste["collection"].isna(), "collection"] = svi_waste["img_dir"].map(collection_from_img_dir)
svi_lookup = build_name_lookup(svi_waste["img_name"])

train_cmp = train_df.drop(columns=["collection_from_dir"], errors="ignore").copy()
train_cmp["svi_img_name"] = train_cmp["img_name"].map(lambda n: match_lookup_name(n, svi_lookup))
train_cmp["svi_waste"] = train_cmp["svi_img_name"].notna()
train_cmp["in_training"] = True
train_cmp["sensitivity_role"] = train_cmp.apply(lambda r: sensitivity_role(True, r["class"], r["svi_waste"]), axis=1)

matched_svi_names = set(train_cmp["svi_img_name"].dropna())
svi_unused = svi_waste[~svi_waste["img_name"].isin(matched_svi_names)].copy()
svi_unused["class"] = "waste"
svi_unused["in_training"] = False
svi_unused["svi_waste"] = True
svi_unused["svi_img_name"] = svi_unused["img_name"]
svi_unused["sensitivity_role"] = "svi_unused_waste"

SHARED_COLS = [
    "img_name", "year", "month", "lat", "lon", "panoid", "img_dir", "exist",
    "collection", "class", "in_training", "svi_waste", "svi_img_name", "sensitivity_role",
]
sensitivity_df = pd.concat([train_cmp[SHARED_COLS], svi_unused[SHARED_COLS]], ignore_index=True)

correct_waste_names = set(svi_waste["img_name"])
waste_extra = svi_unused.copy()
waste_extra["svi_img_name"] = waste_extra["img_name"]
waste_extra["sensitivity_role"] = "correct_svi_waste"

combined = pd.read_csv(COMBINED_SVI_CSV, low_memory=False)
combined["collection"] = combined["img_name"].map(collection_from_name)
combined.loc[combined["collection"].isna(), "collection"] = combined["img_dir"].map(collection_from_img_dir)
background_extra = combined[
    ~combined["img_name"].isin(correct_waste_names) & ~combined["img_name"].isin(train_cmp["img_name"])
].copy()
background_extra["class"] = "background"
background_extra["svi_waste"] = False
background_extra["svi_img_name"] = pd.NA
background_extra["in_training"] = False
background_extra["sensitivity_role"] = "combined_svi_background"

extras = pd.concat([waste_extra[SHARED_COLS], background_extra[SHARED_COLS]], ignore_index=True).drop_duplicates("img_name", keep="first")

POOL_DEFS = [
    ("gsvi_waste", "GSVI", "waste"),
    ("gsvi_background", "GSVI", "background"),
    ("sc_waste", "SC", "waste"),
    ("sc_background", "SC", "background"),
]
POOL_SOURCES = {
    "gsvi_waste": "Correct_SVI.csv",
    "gsvi_background": "Combined_SVI.csv (not in Correct_SVI)",
    "sc_waste": "Correct_SVI.csv",
    "sc_background": "Combined_SVI.csv (not in Correct_SVI)",
}

candidate_pools = {}
for pool_name, collection, cls in POOL_DEFS:
    pool = build_candidate_pool(collection, cls, train_cmp, extras)
    pool["pool"] = pool_name
    candidate_pools[pool_name] = pool

pools_summary = pd.DataFrame({
    "pool": [name for name, _, _ in POOL_DEFS],
    "source": [POOL_SOURCES[name] for name, _, _ in POOL_DEFS],
    "collection": [c for _, c, _ in POOL_DEFS],
    "class": [cls for _, _, cls in POOL_DEFS],
    "in_training": [(candidate_pools[name]["candidate_source"] == "training").sum() for name, _, _ in POOL_DEFS],
    "extra_candidates": [(candidate_pools[name]["candidate_source"] == "extra").sum() for name, _, _ in POOL_DEFS],
    "total": [len(candidate_pools[name]) for name, _, _ in POOL_DEFS],
    "local": [candidate_pools[name]["has_local"].sum() for name, _, _ in POOL_DEFS],
})

gsvi_core = train_cmp[train_cmp["collection"] == "GSVI"].copy()
sc_waste_training = train_cmp[(train_cmp["collection"] == "SC") & (train_cmp["class"] == "waste")].copy()
sc_bg_training = train_cmp[(train_cmp["collection"] == "SC") & (train_cmp["class"] == "background")].copy()
SC_WASTE_SLOTS = len(sc_waste_training)
SC_BG_SLOTS = len(sc_bg_training)
SC_SLOTS = SC_WASTE_SLOTS + SC_BG_SLOTS
GSVI_CORE_SLOTS = len(gsvi_core)

core_names = set(gsvi_core["img_name"])
replacement_pools = {
    "gsvi_waste": waste_extra[(waste_extra["collection"] == "GSVI") & (~waste_extra["img_name"].isin(core_names))].copy(),
    "gsvi_background": background_extra[(background_extra["collection"] == "GSVI") & (~background_extra["img_name"].isin(core_names))].copy(),
}
for pool_df in replacement_pools.values():
    pool_df["has_local"] = pool_df["img_name"].map(has_local_files)

print(f"GSVI core: {GSVI_CORE_SLOTS:,} | SC slots: {SC_SLOTS:,} ({SC_WASTE_SLOTS:,} waste + {SC_BG_SLOTS:,} background)")
print(f"Target: {TOTAL_IMAGES:,} images ({WASTE_TOTAL:,} waste + {BG_TOTAL:,} background)")
print(pools_summary.to_string(index=False))

sensitivity_df.to_csv(SENSITIVITY_CSV, index=False)
pools_summary.to_csv(POOLS_SUMMARY_CSV, index=False)
print(f"Saved {SENSITIVITY_CSV}")
print(f"Saved {POOLS_SUMMARY_CSV}")


GSVI core: 610 | SC slots: 85 (38 waste + 47 background)
Target: 695 images (353 waste + 342 background)
           pool                                source collection      class  in_training  extra_candidates  total  local
     gsvi_waste                       Correct_SVI.csv       GSVI      waste          315              3595   3910    315
gsvi_background Combined_SVI.csv (not in Correct_SVI)       GSVI background          295            449173 449468    295
       sc_waste                       Correct_SVI.csv         SC      waste           38               154    192     38
  sc_background Combined_SVI.csv (not in Correct_SVI)         SC background           47              3516   3563     47
Saved /Users/wenlanzhang/Downloads/PhD_UCL/Data/Chapter_waste/6_sensitivity/Train0315_695_sensitivity_inventory.csv
Saved /Users/wenlanzhang/Downloads/PhD_UCL/Data/Chapter_waste/6_sensitivity/Train0315_695_pools_summary.csv


## 7. GSVI images to label for sensitivity swaps


In [19]:
swap_needs_rows = []
for pct in SC_SHARE_PCTS:
    counts = sc_slot_counts(pct)
    swap_needs_rows.append({
        "scenario": f"sc_{pct}pct",
        "sc_share_pct": pct,
        "sc_waste_swaps": counts["sc_waste_swap"],
        "sc_background_swaps": counts["sc_bg_swap"],
        "gsvi_total_swaps": counts["sc_total_swap"],
    })

swap_needs_df = pd.DataFrame(swap_needs_rows)
max_waste_swaps = int(swap_needs_df["sc_waste_swaps"].max())
max_bg_swaps = int(swap_needs_df["sc_background_swaps"].max())
display(swap_needs_df)

required_w = pick_from_pool(replacement_pools["gsvi_waste"], max_waste_swaps, RANDOM_SEED, "gsvi_waste_required")
required_w["label_role"] = "required"
required_b = pick_from_pool(replacement_pools["gsvi_background"], max_bg_swaps, RANDOM_SEED, "gsvi_background_required")
required_b["label_role"] = "required"
required_replacements = pd.concat([required_w, required_b], ignore_index=True).sort_values(["class", "img_name"]).reset_index(drop=True)

label_waste_n = buffered_count(max_waste_swaps, LABEL_BUFFER_PCT)
label_bg_n = buffered_count(max_bg_swaps, LABEL_BUFFER_PCT)
required_names = set(required_replacements["img_name"])
buffer_parts = []

if label_waste_n > max_waste_swaps:
    waste_pool = replacement_pools["gsvi_waste"][~replacement_pools["gsvi_waste"]["img_name"].isin(required_names)]
    buffer_w = pick_from_pool(waste_pool, label_waste_n - max_waste_swaps, RANDOM_SEED + 1, "gsvi_waste_buffer")
    buffer_w["label_role"] = "buffer"
    buffer_parts.append(buffer_w)

if label_bg_n > max_bg_swaps:
    bg_pool = replacement_pools["gsvi_background"][~replacement_pools["gsvi_background"]["img_name"].isin(required_names)]
    buffer_b = pick_from_pool(bg_pool, label_bg_n - max_bg_swaps, RANDOM_SEED + 1, "gsvi_background_buffer")
    buffer_b["label_role"] = "buffer"
    buffer_parts.append(buffer_b)

label_set = pd.concat([required_replacements] + buffer_parts, ignore_index=True)

label_rows = []
for _, row in label_set.iterrows():
    src = archive_image_path(row["img_name"], row["img_dir"])
    dest_dir = SENSITIVITY_WASTE_DIR if row["class"] == "waste" else SENSITIVITY_BG_DIR
    label_rows.append({
        "img_name": row["img_name"],
        "class": row["class"],
        "label_role": row["label_role"],
        "collection": row["collection"],
        "img_dir": row["img_dir"],
        "src_path": str(src),
        "dest_path": str(dest_dir / f"{row['img_name']}.jpg"),
        "src_exists": src.exists(),
    })

label_manifest_df = pd.DataFrame(label_rows)
swap_needs_df.to_csv(SWAP_NEEDS_CSV, index=False)
label_manifest_df.to_csv(LABEL_MANIFEST_CSV, index=False)
print(f"Label set: {len(label_set):,} images ({label_waste_n:,} waste + {label_bg_n:,} background incl. buffer)")
print(f"Source found: {label_manifest_df['src_exists'].sum():,} / {len(label_manifest_df):,}")
print(f"Saved {SWAP_NEEDS_CSV}")
print(f"Saved {LABEL_MANIFEST_CSV}")

if COPY_IMAGES:
    SENSITIVITY_WASTE_DIR.mkdir(parents=True, exist_ok=True)
    SENSITIVITY_BG_DIR.mkdir(parents=True, exist_ok=True)
    copied, skipped = 0, 0
    for _, row in label_manifest_df.iterrows():
        src, dest = Path(row["src_path"]), Path(row["dest_path"])
        if not src.exists() or dest.exists():
            skipped += 1
            continue
        shutil.copy2(src, dest)
        copied += 1
    print(f"Copied {copied:,} | skipped {skipped:,}")
else:
    print("COPY_IMAGES = False — set True to copy images")


,scenario,sc_share_pct,sc_waste_swaps,sc_background_swaps,gsvi_total_swaps
0,sc_0pct,0,38,47,85
1,sc_25pct,25,28,35,63
2,sc_50pct,50,19,23,42
3,sc_75pct,75,10,12,22
4,sc_100pct,100,0,0,0


Label set: 120 images (54 waste + 66 background incl. buffer)
Source found: 120 / 120
Saved /Users/wenlanzhang/Downloads/PhD_UCL/Data/Chapter_waste/6_sensitivity/Train0315_695_gsvi_swap_needs.csv
Saved /Users/wenlanzhang/Downloads/PhD_UCL/Data/Chapter_waste/6_sensitivity/Train0315_695_gsvi_label_manifest.csv
COPY_IMAGES = False — set True to copy images


## 8. Build scenario training folders (post-labelling)

**Labelled replacement pools:**
- **Waste** → `20260708_label_waste/images/` + `20260708_label_waste/labels/` (non-empty `.txt` = waste)
- **Background** → `20260708_sensitivity/background/` (no label files; scenario build writes empty `.txt`)

Images in the waste export with missing/empty labels are treated as background.


In [20]:
def resolve_labeled_label_path(img_stem: str, label_dir: Path) -> Path | None:
    exact = label_dir / f"{img_stem}.txt"
    if exact.exists():
        return exact
    matches = sorted(label_dir.glob(f"*-{img_stem}.txt"))
    return matches[0] if matches else None


def resolve_train_files(img_name: str) -> tuple[Path, Path, str]:
    for cand in candidate_names(img_name):
        img = IMAGES_DIR / f"{cand}.jpg"
        lbl = LABELS_DIR / f"{cand}.txt"
        if img.exists() and lbl.exists():
            return img, lbl, cand
    tried_imgs = [IMAGES_DIR / f"{c}.jpg" for c in candidate_names(img_name)]
    tried_lbls = [LABELS_DIR / f"{c}.txt" for c in candidate_names(img_name)]
    raise FileNotFoundError(
        f"Missing Train0315_695 files for '{img_name}'.\n"
        f"  Tried images: {[str(p) for p in tried_imgs]}\n"
        f"  Tried labels: {[str(p) for p in tried_lbls]}"
    )


def load_labeled_replacement_pools() -> dict[str, pd.DataFrame]:
    """Waste from Label Studio export; background from sensitivity/background (empty labels)."""
    waste_rows, bg_rows = [], []

    if not LABELED_WASTE_DIR.exists():
        raise FileNotFoundError(f"Label Studio waste images not found: {LABELED_WASTE_DIR}")
    if not LABELED_WASTE_LABEL_DIR.exists():
        raise FileNotFoundError(f"Label Studio waste labels not found: {LABELED_WASTE_LABEL_DIR}")

    for img_path in sorted(LABELED_WASTE_DIR.glob("*.jpg")):
        file_stem = img_path.stem
        label_path = resolve_labeled_label_path(file_stem, LABELED_WASTE_LABEL_DIR)
        has_waste = label_path is not None and label_path.stat().st_size > 0
        out_stem = label_path.stem if label_path is not None else file_stem
        row = {
            "img_name": canonical_img_name(file_stem),
            "file_stem": out_stem,
            "class": "waste" if has_waste else "background",
            "collection": "GSVI",
            "image_path": str(img_path),
            "label_path": str(label_path) if has_waste else None,
            "has_local": True,
            "candidate_source": "labeled",
        }
        (waste_rows if has_waste else bg_rows).append(row)

    if not LABELED_BG_DIR.exists():
        raise FileNotFoundError(f"Background batch not found: {LABELED_BG_DIR}")

    for img_path in sorted(LABELED_BG_DIR.glob("*.jpg")):
        file_stem = img_path.stem
        bg_rows.append({
            "img_name": canonical_img_name(file_stem),
            "file_stem": file_stem,
            "class": "background",
            "collection": "GSVI",
            "image_path": str(img_path),
            "label_path": None,  # empty label written in materialize_scenario_folder
            "has_local": True,
            "candidate_source": "background_batch",
        })

    return {
        "gsvi_waste": pd.DataFrame(waste_rows).drop_duplicates("img_name"),
        "gsvi_background": pd.DataFrame(bg_rows).drop_duplicates("img_name"),
    }


def build_labeled_lookup(pools: dict[str, pd.DataFrame]) -> dict[str, dict]:
    lookup: dict[str, dict] = {}
    for pool_df in pools.values():
        for _, row in pool_df.iterrows():
            lookup[row["img_name"]] = row.to_dict()
    return lookup


def build_swap_scenario_labeled(sc_share_pct: int, labeled_pools: dict[str, pd.DataFrame], seed: int = RANDOM_SEED) -> pd.DataFrame:
    counts = sc_slot_counts(sc_share_pct)
    parts = [gsvi_core.assign(swap_role="gsvi_core")]
    parts.append(pick_training_rows(sc_waste_training, counts["sc_waste_keep"], seed).assign(swap_role="sc_kept"))
    parts.append(pick_training_rows(sc_bg_training, counts["sc_bg_keep"], seed).assign(swap_role="sc_kept"))
    if counts["sc_waste_swap"]:
        parts.append(pick_from_pool(labeled_pools["gsvi_waste"], counts["sc_waste_swap"], seed, "labeled_gsvi_waste").assign(swap_role="gsvi_replacement"))
    if counts["sc_bg_swap"]:
        parts.append(pick_from_pool(labeled_pools["gsvi_background"], counts["sc_bg_swap"], seed, "labeled_gsvi_background").assign(swap_role="gsvi_replacement"))
    scenario = pd.concat(parts, ignore_index=True)
    scenario["sc_share_pct"] = sc_share_pct
    scenario["scenario"] = f"sc_{sc_share_pct}pct"
    assert len(scenario) == TOTAL_IMAGES
    assert (scenario["class"] == "waste").sum() == WASTE_TOTAL
    assert (scenario["class"] == "background").sum() == BG_TOTAL
    return scenario


def materialize_scenario_folder(scenario_df: pd.DataFrame, scenario_dir: Path, labeled_lookup: dict[str, dict]) -> int:
    images_dir = scenario_dir / "images"
    labels_dir = scenario_dir / "labels"
    if scenario_dir.exists():
        shutil.rmtree(scenario_dir)
    images_dir.mkdir(parents=True)
    labels_dir.mkdir(parents=True)

    train_paths: list[str] = []
    for _, row in scenario_df.iterrows():
        if row["swap_role"] in ("gsvi_core", "sc_kept"):
            src_img, src_lbl, out_stem = resolve_train_files(row["img_name"])
        else:
            if row["img_name"] not in labeled_lookup:
                raise KeyError(
                    f"Replacement '{row['img_name']}' not in labelled lookup "
                    f"(scenario={scenario_dir.name}, swap_role={row['swap_role']}).\n"
                    f"  Check images: {LABELED_WASTE_DIR} / {LABELED_BG_DIR}\n"
                    f"  Check labels: {LABELED_WASTE_LABEL_DIR}"
                )
            info = labeled_lookup[row["img_name"]]
            src_img = Path(info["image_path"])
            if not src_img.exists():
                raise FileNotFoundError(
                    f"Replacement image missing for '{row['img_name']}'.\n"
                    f"  Expected: {src_img}\n"
                    f"  scenario: {scenario_dir.name}"
                )
            label_path = info.get("label_path")
            src_lbl = Path(label_path) if label_path else None
            out_stem = info["file_stem"]

        dest_img = images_dir / f"{out_stem}.jpg"
        dest_lbl = labels_dir / f"{out_stem}.txt"
        shutil.copy2(src_img, dest_img)
        if src_lbl and src_lbl.exists():
            shutil.copy2(src_lbl, dest_lbl)
        else:
            dest_lbl.write_text("")
        train_paths.append(str(dest_img.resolve()))

    train_txt = scenario_dir / "train.txt"
    train_txt.write_text("\n".join(train_paths) + "\n")
    (scenario_dir / "Waste.yaml").write_text(f"train: {train_txt}\nval: {train_txt}\n\nnc: 1\n\nnames: ['Waste']\n")
    return len(train_paths)


labeled_pools = load_labeled_replacement_pools()
labeled_lookup = build_labeled_lookup(labeled_pools)

n_waste_imgs = len(list(LABELED_WASTE_DIR.glob("*.jpg"))) if LABELED_WASTE_DIR.exists() else 0
n_waste_pool = len(labeled_pools["gsvi_waste"])
if n_waste_imgs and n_waste_pool == 0:
    raise RuntimeError(
        f"Found {n_waste_imgs} waste images in {LABELED_WASTE_DIR}, "
        f"but 0 classified as waste (empty gsvi_waste pool).\n"
        f"  Labels dir: {LABELED_WASTE_LABEL_DIR}\n"
        f"  Exists: {LABELED_WASTE_LABEL_DIR.exists()}\n"
        f"  Export non-empty .txt label files from Label Studio before building scenarios."
    )

print(f"Labelled pools: waste={n_waste_pool:,}, background={len(labeled_pools['gsvi_background']):,}")
if not LABELED_WASTE_LABEL_DIR.exists():
    print(f"WARNING: labels dir missing — {LABELED_WASTE_LABEL_DIR}")

scenario_tables = {}
scenario_rows = []
baseline_names = set(train_cmp["img_name"])

for pct in SC_SHARE_PCTS:
    tag = f"sc_{pct}pct"
    scenario_df = build_swap_scenario_labeled(pct, labeled_pools)
    scenario_tables[tag] = scenario_df
    counts = sc_slot_counts(pct)
    scenario_names = set(scenario_df["img_name"])
    scenario_rows.append({
        "scenario": tag,
        "sc_share_pct": pct,
        "sc_kept": counts["sc_total_keep"],
        "sc_swapped_to_gsvi": counts["sc_total_swap"],
        "gsvi_images": (scenario_df["collection"] == "GSVI").sum(),
        "sc_images": (scenario_df["collection"] == "SC").sum(),
        "gsvi_replacements": (scenario_df["swap_role"] == "gsvi_replacement").sum(),
        "total_images": len(scenario_df),
        "swapped_out": len(baseline_names - scenario_names),
        "swapped_in": len(scenario_names - baseline_names),
    })

scenario_summary = pd.DataFrame(scenario_rows)
display(scenario_summary)

if BUILD_SCENARIOS:
    SCENARIO_DIR.mkdir(parents=True, exist_ok=True)
    for tag, scenario_df in scenario_tables.items():
        scenario_dir = SENSITIVITY_IMG_ROOT / tag
        n_images = materialize_scenario_folder(scenario_df, scenario_dir, labeled_lookup)
        scenario_df.to_csv(SCENARIO_DIR / f"{tag}_training.csv", index=False)
        scenario_df.to_csv(scenario_dir / "training.csv", index=False)
        print(f"{tag}: {n_images:,} images → {scenario_dir}")
    scenario_summary.to_csv(SCENARIO_DIR / "scenario_summary.csv", index=False)
else:
    print("BUILD_SCENARIOS = False")


Labelled pools: waste=39, background=62


,scenario,sc_share_pct,sc_kept,sc_swapped_to_gsvi,gsvi_images,sc_images,gsvi_replacements,total_images,swapped_out,swapped_in
0,sc_0pct,0,0,85,695,0,85,695,85,85
1,sc_25pct,25,22,63,673,22,63,695,63,63
2,sc_50pct,50,43,42,652,43,42,695,42,42
3,sc_75pct,75,63,22,632,63,22,695,22,22
4,sc_100pct,100,85,0,610,85,0,695,0,0


sc_0pct: 695 images → /Users/wenlanzhang/Downloads/PhD_UCL/Data/Waste/img/sensitivity/sc_0pct
sc_25pct: 695 images → /Users/wenlanzhang/Downloads/PhD_UCL/Data/Waste/img/sensitivity/sc_25pct
sc_50pct: 695 images → /Users/wenlanzhang/Downloads/PhD_UCL/Data/Waste/img/sensitivity/sc_50pct
sc_75pct: 695 images → /Users/wenlanzhang/Downloads/PhD_UCL/Data/Waste/img/sensitivity/sc_75pct
sc_100pct: 695 images → /Users/wenlanzhang/Downloads/PhD_UCL/Data/Waste/img/sensitivity/sc_100pct


## 9. Panoid-safe held-out test set


In [21]:
def build_training_exclusion_sets(train_df_in: pd.DataFrame) -> tuple[set[str], set[str], dict[str, str]]:
    names = set(train_df_in["img_name"])
    lookup = build_name_lookup(train_df_in["img_name"])
    panoids = set(train_df_in.loc[train_df_in["collection"] == "GSVI", "panoid"].dropna().astype(str))
    return names, panoids, lookup


def is_in_training(img_name: str, training_names: set[str], training_lookup: dict[str, str]) -> bool:
    return img_name in training_names or match_lookup_name(img_name, training_lookup) is not None


def build_panoid_safe_units(candidate_df: pd.DataFrame, gsvi_only: bool = False) -> pd.DataFrame:
    rows = []
    if gsvi_only:
        for panoid, grp in candidate_df.groupby("panoid", sort=True):
            unit_classes = set(grp["class"])
            if len(unit_classes) > 1:
                continue
            rows.append({
                "unit_id": f"GSVI:{panoid}",
                "collection": "GSVI",
                "label_class": "waste" if "waste" in unit_classes else "background",
                "n_images": len(grp),
                "img_names": sorted(grp["img_name"].tolist()),
                "panoids": [str(panoid)],
            })
    else:
        unit_key = candidate_df.apply(
            lambda r: f"GSVI:{r['panoid']}" if r["collection"] == "GSVI" else f"SC:{r['img_name']}", axis=1
        )
        for unit_id, grp in candidate_df.groupby(unit_key, sort=True):
            unit_classes = set(grp["class"])
            if grp["collection"].iloc[0] == "GSVI" and len(unit_classes) > 1:
                continue
            rows.append({
                "unit_id": unit_id,
                "collection": grp["collection"].iloc[0],
                "label_class": "waste" if "waste" in unit_classes else "background",
                "n_images": len(grp),
                "img_names": sorted(grp["img_name"].tolist()),
                "panoids": sorted({str(p) for p in grp["panoid"].dropna()}),
            })
    return pd.DataFrame(rows)


def build_external_candidate_pool(training_names, training_panoids, training_lookup) -> pd.DataFrame:
    parts = [
        replacement_pools["gsvi_waste"],
        replacement_pools["gsvi_background"],
        extras[(extras["collection"] == "SC") & (extras["class"] == "waste")],
        extras[(extras["collection"] == "SC") & (extras["class"] == "background")],
    ]
    pool = pd.concat(parts, ignore_index=True).drop_duplicates("img_name", keep="first")
    pool = pool[~pool["img_name"].map(lambda n: is_in_training(n, training_names, training_lookup))]
    gsvi_mask = pool["collection"] == "GSVI"
    pool = pool[~gsvi_mask | ~pool["panoid"].astype(str).isin(training_panoids)]
    pool["source_image"] = pool.apply(lambda r: archive_image_path(r["img_name"], r["img_dir"]), axis=1)
    pool["image_exists"] = pool["source_image"].map(Path.exists)
    return pool[pool["image_exists"]].reset_index(drop=True)


def select_external_units(unit_df: pd.DataFrame, target_images: int, seed: int, waste_share: float = 0.5) -> pd.DataFrame:
    if unit_df.empty:
        raise ValueError("No external units available")
    target_waste = round(target_images * waste_share)
    target_background = target_images - target_waste
    pool = unit_df.sample(frac=1, random_state=seed).reset_index(drop=True)
    picked_ids: set[str] = set()
    picked_rows: list[pd.Series] = []

    def pick_class(label_class: str, target: int) -> int:
        images = 0
        for _, row in pool.iterrows():
            if row["unit_id"] in picked_ids or row["label_class"] != label_class:
                continue
            picked_rows.append(row)
            picked_ids.add(row["unit_id"])
            images += row["n_images"]
            if images >= target:
                break
        return images

    waste_images = pick_class("waste", target_waste)
    bg_images = pick_class("background", target_background)
    for label_class, target, got in [("waste", target_waste, waste_images), ("background", target_background, bg_images)]:
        if got >= target:
            continue
        for _, row in pool.iterrows():
            if row["unit_id"] in picked_ids or row["label_class"] != label_class:
                continue
            picked_rows.append(row)
            picked_ids.add(row["unit_id"])
            got += row["n_images"]
            if got >= target:
                break
    return pd.DataFrame(picked_rows).reset_index(drop=True)


def export_held_out_images(test_df: pd.DataFrame, out_dir: Path) -> tuple[int, pd.DataFrame]:
    waste_dir = out_dir / "waste"
    bg_dir = out_dir / "background"
    if out_dir.exists():
        shutil.rmtree(out_dir)
    waste_dir.mkdir(parents=True)
    bg_dir.mkdir(parents=True)

    manifest_rows = []
    copied = 0
    for _, row in test_df.sort_values("img_name").iterrows():
        src_img = archive_image_path(row["img_name"], row["img_dir"])
        if not src_img.exists():
            raise FileNotFoundError(
                f"Missing held-out source image.\n"
                f"  img_name: {row['img_name']}\n"
                f"  expected path: {src_img}\n"
                f"  img_dir: {row.get('img_dir')}\n"
                f"  unit_id: {row.get('unit_id')}\n"
                f"  dest folder: {out_dir}"
            )
        dest_dir = waste_dir if row["class"] == "waste" else bg_dir
        dest_img = dest_dir / f"{row['img_name']}.jpg"
        shutil.copy2(src_img, dest_img)
        copied += 1
        manifest_rows.append({
            "img_name": row["img_name"],
            "class": row["class"],
            "collection": row["collection"],
            "unit_id": row["unit_id"],
            "panoid": row.get("panoid"),
            "img_dir": row["img_dir"],
            "src_path": str(src_img),
            "dest_path": str(dest_img),
            "label_status": "manual_required" if row["class"] == "waste" else "background_no_label",
        })
    return copied, pd.DataFrame(manifest_rows)


training_names, training_panoids, training_lookup = build_training_exclusion_sets(train_cmp)
external_pool = build_external_candidate_pool(training_names, training_panoids, training_lookup)
external_units = build_panoid_safe_units(external_pool)
selected_units = select_external_units(external_units, TEST_TARGET_IMAGES, HELD_OUT_SEED, waste_share=WASTE_SHARE)

selected_names = sorted({n for names in selected_units["img_names"] for n in names})
held_out_df = external_pool[external_pool["img_name"].isin(selected_names)].copy()
held_out_df["unit_id"] = held_out_df.apply(
    lambda r: f"GSVI:{r['panoid']}" if r["collection"] == "GSVI" else f"SC:{r['img_name']}", axis=1
)
held_out_df["in_training_695"] = False
held_out_df["held_out_test"] = True

print(f"Held-out: {len(held_out_df):,} images from {len(selected_units):,} units")
print(held_out_df.groupby(["collection", "class"]).size().unstack(fill_value=0))

held_out_df.to_csv(HELD_OUT_CSV, index=False)
selected_units.to_csv(HELD_OUT_UNITS_CSV, index=False)

if BUILD_HELD_OUT:
    n_images, copy_manifest_df = export_held_out_images(held_out_df, HELD_OUT_DIR)
    copy_manifest_df.to_csv(HELD_OUT_MANIFEST_CSV, index=False)
    print(f"{HELD_OUT_TAG}: {n_images:,} images → {HELD_OUT_DIR}")
else:
    print("BUILD_HELD_OUT = False")


Held-out: 107 images from 59 units
class       background  waste
collection                   
GSVI                52     12
SC                   0     43
20260714_heldout: 107 images → /Users/wenlanzhang/Downloads/PhD_UCL/Data/Waste/img/sensitivity/20260714_heldout


## 10. GSVI-only held-out test set


In [22]:
def cap_balanced_target(units_df: pd.DataFrame, target_images: int) -> int:
    by_class = units_df.groupby("label_class")["n_images"].sum().reindex(["waste", "background"], fill_value=0)
    max_balanced = int(2 * min(by_class["waste"], by_class["background"]))
    capped = min(target_images, max_balanced)
    if capped < target_images:
        print(f"WARNING: requested {target_images:,} but balanced GSVI pool supports {max_balanced:,}")
    return capped


def build_gsvi_external_pool(training_names, training_panoids, training_lookup, exclude_names=None, exclude_panoids=None) -> pd.DataFrame:
    pool = pd.concat([replacement_pools["gsvi_waste"], replacement_pools["gsvi_background"]], ignore_index=True).drop_duplicates("img_name", keep="first")
    pool = pool[pool["collection"] == "GSVI"].copy()
    pool = pool[~pool["img_name"].map(lambda n: is_in_training(n, training_names, training_lookup))]
    pool = pool[~pool["panoid"].astype(str).isin(training_panoids)]
    if exclude_names:
        pool = pool[~pool["img_name"].isin(exclude_names)]
    if exclude_panoids:
        pool = pool[~pool["panoid"].astype(str).isin(exclude_panoids)]
    pool["source_image"] = pool.apply(lambda r: archive_image_path(r["img_name"], r["img_dir"]), axis=1)
    pool["image_exists"] = pool["source_image"].map(Path.exists)
    return pool[pool["image_exists"]].reset_index(drop=True)


prior_names: set[str] = set()
prior_panoids: set[str] = set()
if EXCLUDE_PRIOR_HELD_OUT and PRIOR_HELD_OUT_CSV.exists():
    prior_df = pd.read_csv(PRIOR_HELD_OUT_CSV)
    prior_names = set(prior_df["img_name"])
    prior_panoids = set(prior_df.loc[prior_df["collection"] == "GSVI", "panoid"].dropna().astype(str))

gsvi_pool = build_gsvi_external_pool(training_names, training_panoids, training_lookup, prior_names, prior_panoids)
gsvi_units = build_panoid_safe_units(gsvi_pool, gsvi_only=True)
effective_target = cap_balanced_target(gsvi_units, GSVI_TEST_TARGET_IMAGES)
selected_gsvi_units = select_external_units(gsvi_units, effective_target, HELD_OUT_GSVI_SEED, waste_share=GSVI_WASTE_SHARE)

selected_gsvi_names = sorted({n for names in selected_gsvi_units["img_names"] for n in names})
held_out_gsvi_df = gsvi_pool[gsvi_pool["img_name"].isin(selected_gsvi_names)].copy()
held_out_gsvi_df["unit_id"] = held_out_gsvi_df["panoid"].map(lambda p: f"GSVI:{p}")
held_out_gsvi_df["in_training_695"] = False
held_out_gsvi_df["held_out_gsvi_test"] = True

print(f"GSVI held-out: {len(held_out_gsvi_df):,} images from {len(selected_gsvi_units):,} units")
print(held_out_gsvi_df["class"].value_counts().to_string())

held_out_gsvi_df.to_csv(HELD_OUT_GSVI_CSV, index=False)
selected_gsvi_units.to_csv(HELD_OUT_GSVI_UNITS_CSV, index=False)

if BUILD_HELD_OUT_GSVI:
    n_images, copy_manifest_df = export_held_out_images(held_out_gsvi_df, HELD_OUT_GSVI_DIR)
    copy_manifest_df.to_csv(HELD_OUT_GSVI_MANIFEST_CSV, index=False)
    print(f"{HELD_OUT_GSVI_TAG}: {n_images:,} images → {HELD_OUT_GSVI_DIR}")
else:
    print("BUILD_HELD_OUT_GSVI = False")


GSVI held-out: 64 images from 16 units
class
waste         32
background    32
20260714_heldout_GSVI: 64 images → /Users/wenlanzhang/Downloads/PhD_UCL/Data/Waste/img/sensitivity/20260714_heldout_GSVI
